# Week 64, Day 1: How CPython runs your code

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w064d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import ast

tree = ast.parse("total = price * 2")
assign = tree.body[0]
print(type(assign).__name__)
print(assign.targets[0].id, type(assign.value).__name__)
print(ast.unparse(assign.value))

code = compile("price * 2", "<calc>", "eval")
print(eval(code, {"price": 21}))
print(type(code).__name__, code.co_names)

In [ ]:
import ast, io, tokenize

source = "if n > 0:\n    n = n - 1\n"
for tok in tokenize.generate_tokens(io.StringIO(source).readline):
    print(tokenize.tok_name[tok.type], repr(tok.string))

tree = ast.parse(source)
print(type(tree.body[0]).__name__, type(tree.body[0].test).__name__)

code = compile(tree, "<example>", "exec")
namespace = {"n": 3}
exec(code, namespace)
print(namespace["n"])

In [ ]:
import io, keyword, tokenize

source = "def area(r):  # circle\n    return 3.14 * r ** 2\n"
names, keywords, comments = [], [], []
for tok in tokenize.generate_tokens(io.StringIO(source).readline):
    if tok.type == tokenize.NAME:
        (keywords if keyword.iskeyword(tok.string) else names).append(tok.string)
    elif tok.type == tokenize.COMMENT:
        comments.append((tok.start, tok.string))
print(names, keywords, comments)

In [ ]:
import ast

source = """
def total(prices, tax):
    subtotal = sum(prices)
    return round(subtotal * (1 + tax), 2)

def label(item):
    return item.upper()
"""

class CallCounter(ast.NodeVisitor):
    def __init__(self):
        self.calls = []
    def visit_FunctionDef(self, node):
        print("function", node.name, "args", [a.arg for a in node.args.args])
        self.generic_visit(node)             # carry on into the body
    def visit_Call(self, node):
        self.calls.append(ast.unparse(node.func))
        self.generic_visit(node)

counter = CallCounter()
counter.visit(ast.parse(source))
print(counter.calls)
print(ast.literal_eval("{'a': [1, 2.5], 'b': (True, None)}"))

In [ ]:
def make_counter(start):
    count = start
    def step(by=1):
        nonlocal count
        count += by
        return count
    return step

step = make_counter(10)
code = step.__code__
print(code.co_name, code.co_argcount, code.co_varnames, code.co_freevars)
outer = make_counter.__code__
inner_codes = [c for c in outer.co_consts if hasattr(c, "co_name")]
print([c.co_name for c in inner_codes], inner_codes[0] is code)

try:
    compile("x = ", "<typed in>", "exec")
except SyntaxError as e:
    print("SyntaxError in", e.filename, "line", e.lineno)

namespace = {}
exec(compile("a = 2\nb = a * 21", "<string>", "exec"), namespace)
print(namespace["b"], eval(compile("a + b", "<string>", "eval"), namespace))

In [ ]:
import importlib.util, marshal

print(importlib.util.cache_from_source("shop.py").replace("\\", "/").split("/")[0])

code = compile("print('loaded from bytes')", "<cached>", "exec")
data = marshal.dumps(code)          # what goes into a .pyc after its header
again = marshal.loads(data)
exec(again)
print(type(data).__name__, again.co_filename)

> **Key idea.** CPython reads source, tokenises it (tokenize), parses tokens into an abstract syntax tree (ast), builds a symbol table, compiles the tree to bytecode in a code object (compile), and runs it in the evaluation loop. Tokens keep comments and positions; the AST keeps structure: walk it with ast.walk, analyse it with NodeVisitor (calling generic_visit to go deeper), rewrite it with NodeTransformer plus fix_missing_locations, and turn it back into text with ast.unparse. compile's modes are exec (statements), eval (one expression) and single (the REPL). Code objects hold bytecode, constants, names and local names; every function has one in __code__. Imported modules' code objects are cached in __pycache__ as version-specific .pyc files.

### Exercise: Counting tokens

Write `token_counts(source)`: a dict from token type name (`"NAME"`, `"OP"`, `"NUMBER"`, `"STRING"`, ...) to how many tokens of that type the source contains, **leaving out** the layout tokens `NEWLINE`, `NL`, `INDENT`, `DEDENT` and `ENDMARKER`. Then `strip_comments(source)`: the source with every comment removed, using the tokeniser to find them (so a `#` inside a string isn't touched), with any spaces left at the end of each line removed too.

In [ ]:
import io
import tokenize

def token_counts(source):
    ___

def strip_comments(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tokens():\n    \"the tokeniser\"\n    src = \"x = 1 + y  # add\\nif x:\\n    print('# not a comment', x)\\n\"\n    assert token_counts(src) == {\"NAME\": 6, \"OP\": 6, \"NUMBER\": 1, \"COMMENT\": 1, \"STRING\": 1}\n    assert token_counts(\"\") == {}\n    assert strip_comments(src) == \"x = 1 + y\\nif x:\\n    print('# not a comment', x)\\n\"\n    assert strip_comments(\"# only\\ny = 2  # two\\n\") == \"\\ny = 2\\n\"")

### Exercise: What a program uses

Using the AST, write `names_read(source)`: the sorted list of distinct names the code **reads** (a `Name` node with a `Load` context), and `definitions(source)`: a dict with keys `"functions"`, `"classes"` and `"imports"`, each a sorted list: names of every function (including methods and nested functions) and class defined anywhere, and every module imported (`import a.b` gives `"a.b"`, `from c import d` gives `"c"`).

In [ ]:
import ast

def names_read(source):
    ___

def definitions(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_names():\n    \"walking the tree\"\n    src = (\n        \"import os\\nimport collections.abc\\nfrom math import pi\\n\"\n        \"class Shape:\\n    def area(self):\\n        def helper():\\n            return pi\\n        return helper() * self.r\\n\"\n        \"def main():\\n    s = Shape()\\n    print(s.area(), os.sep)\\n\"\n    )\n    assert names_read(src) == [\"Shape\", \"helper\", \"os\", \"pi\", \"print\", \"s\", \"self\"]\n    assert definitions(src) == {\"functions\": [\"area\", \"helper\", \"main\"], \"classes\": [\"Shape\"], \"imports\": [\"collections.abc\", \"math\", \"os\"]}\n    assert names_read(\"x = 1\") == [] and definitions(\"\") == {\"functions\": [], \"classes\": [], \"imports\": []}")

### Exercise: The three modes

Write `kind_of_code(source)`: `"expression"` if the source compiles in `"eval"` mode, otherwise `"statements"` if it compiles in `"exec"` mode, otherwise `"invalid"`. Then `evaluate(source, variables)` that returns the value for an expression and, for statements, a dict of the **new or changed** names they set (compared with `variables`, which must not be modified), ignoring `__builtins__`.

In [ ]:
def kind_of_code(source):
    ___

def evaluate(source, variables):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_modes():\n    \"compile modes\"\n    assert kind_of_code(\"1 + 2\") == \"expression\" and kind_of_code(\"x = 1\") == \"statements\"\n    assert kind_of_code(\"for i in range(3): pass\") == \"statements\" and kind_of_code(\"x = \") == \"invalid\"\n    assert kind_of_code(\"f(x)\") == \"expression\"\n    start = {\"a\": 2, \"b\": 3}\n    assert evaluate(\"a * b\", start) == 6\n    assert evaluate(\"c = a + b\\na = 10\\nb = 3\", start) == {\"c\": 5, \"a\": 10}\n    assert start == {\"a\": 2, \"b\": 3}\n    assert evaluate(\"total = sum(range(4))\", {}) == {\"total\": 6}")

### Exercise: Reading a code object

Write `describe(func)`: a dict built from `func.__code__` with `"name"`, `"positional"` (the names of the ordinary positional parameters, a list), `"locals"` (local variable names that **aren't** parameters, a list in order), `"free"` (a list of the closure variables it uses), `"varargs"` and `"varkw"` (True if it has `*args` / `**kwargs`, from `co_flags` with `inspect.CO_VARARGS` and `inspect.CO_VARKEYWORDS`), and `"nested"` (names of functions defined directly inside it, found in `co_consts`). Keyword-only parameters count as parameters (`co_kwonlyargcount`).

In [ ]:
import inspect

def describe(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_codeobj():\n    \"code objects\"\n    def outer(a, b=2, *rest, key=None, **extra):\n        total = a + b\n        def inner():\n            return total\n        def other():\n            pass\n        return inner\n    d = describe(outer)\n    assert d[\"name\"] == \"outer\" and d[\"positional\"] == [\"a\", \"b\"] and d[\"varargs\"] and d[\"varkw\"]\n    assert set(d[\"locals\"]) == {\"total\", \"inner\", \"other\"} and d[\"nested\"] == [\"inner\", \"other\"]\n    inner = outer(1)\n    di = describe(inner)\n    assert di[\"free\"] == [\"total\"] and di[\"positional\"] == [] and di[\"locals\"] == [] and not di[\"varargs\"]\n    assert describe(lambda x, *, y: x)[\"positional\"] == [\"x\"] and describe(lambda x, *, y: x)[\"locals\"] == []")

### Exercise: Rewriting code

Write two `ast.NodeTransformer`s and functions that apply them and return the new source with `ast.unparse`: `fold(source)` replaces every `+`, `-` or `*` between two **constants** with the result, working from the inside out (so `2 * 3 + 1` becomes `7`), and `rename(source, old, new)` renames every variable, parameter and function called `old` to `new` (`Name` nodes, `arg` nodes and `FunctionDef` names). Check that the result still compiles.

In [ ]:
import ast
import operator

def fold(source):
    ___

def rename(source, old, new):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_transform():\n    \"NodeTransformer\"\n    assert fold(\"x = 2 * 3 + 1\") == \"x = 7\"\n    assert fold(\"y = a * (4 - 1) + 2 * 5\") == \"y = a * 3 + 10\"\n    assert fold(\"z = 'ab' + 'c'\") == \"z = 'abc'\" and fold(\"w = a + b\") == \"w = a + b\"\n    out = rename(\"def total(n):\\n    total = n + 1\\n    return total\\nprint(total(2))\", \"total\", \"tally\")\n    assert out == \"def tally(n):\\n    tally = n + 1\\n    return tally\\nprint(tally(2))\"\n    assert rename(\"def f(x):\\n    return x * 2\", \"x\", \"value\") == \"def f(value):\\n    return value * 2\"\n    compile(fold(\"q = 1 + 2\"), \"<t>\", \"exec\")")

### Bug hunt: eval for a statement

`eval` only accepts a single expression, so asking it to run an assignment fails with `SyntaxError`. Run the statement with `exec` in the namespace instead, so this prints `30`.

In [ ]:
namespace = {"width": 5, "height": 6}
eval("area = width * height", namespace)
print(namespace["area"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"exec for statements\"\n    expect_output(\"30\")")

### Bug hunt: A visitor that stops at the top

`visit_FunctionDef` doesn't call `generic_visit`, so the visitor never goes inside a function, never finds the nested one, and this prints `1`. Make it carry on into each function's body so it prints `2`.

In [ ]:
import ast

class Functions(ast.NodeVisitor):
    def __init__(self):
        self.count = 0
    def visit_FunctionDef(self, node):
        self.count += 1

source = "def outer():\n    def inner():\n        pass\n    return inner\n"
v = Functions()
v.visit(ast.parse(source))
print(v.count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"generic_visit\"\n    expect_output(\"2\")")

### Bug hunt: New nodes without positions

The transformer replaces each number with a brand-new `Constant` node that has no line number, and compiling a tree with nodes missing their positions fails with `TypeError`. Give the new nodes positions with `ast.fix_missing_locations` before compiling, so this prints `20`.

In [ ]:
import ast

class Double(ast.NodeTransformer):
    def visit_Constant(self, node):
        return ast.Constant(node.value * 2)

tree = Double().visit(ast.parse("result = 4 + 6", mode="exec"))
namespace = {}
exec(compile(tree, "<doubled>", "exec"), namespace)
print(namespace["result"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fix_missing_locations\"\n    expect_output(\"20\")")

### Puzzle: Counting calls in a tree

Arrange the lines of a visitor that counts function calls, including calls nested inside other calls. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the compile mode for a single expression: "exec", "eval" or "single"
q = ___   # type(ast.parse("x + 1").body[0]).__name__
r = ___   # True or False: the AST keeps the program's comments
s = ___   # what exec returns, whatever the code does
t = ___   # True or False: python script.py caches script.py's own bytecode in __pycache__

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"eval\" and q == \"Expr\" and r is False, \"p, q and r\"\n    assert s is None and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a safe calculator

`eval` on text you didn't write is dangerous: `eval("__import__('os').remove('data.txt')")` would do exactly that. Write `safe_eval(expression, names=None)` that evaluates an expression by **walking its AST yourself** and allowing only:

- numbers (`int` and `float` constants; booleans count as numbers);
- names, looked up in the `names` dict (an unknown name raises `NameError`);
- the binary operators `+ - * / // % **`, unary `-` and `+`, and comparisons (`<`, `<=`, `>`, `>=`, `==`, `!=`, including chains such as `0 < x < 10`);
- `and`, `or` and `not`, with Python's short-circuit behaviour;
- conditional expressions (`a if test else b`);
- calls to **functions in `names`**, with positional arguments only.

Anything else (attribute access, strings, subscripts, lambdas, keyword arguments, calls to things not in `names`) raises `ValueError` naming the node type. To keep things bounded, `**` with an exponent above 100 raises `ValueError`. Don't call `eval` or `exec`.

In [ ]:
import ast
import operator

def safe_eval(expression, names=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_arithmetic():\n    \"allowed expressions\"\n    assert safe_eval(\"2 + 3 * 4\") == 14 and safe_eval(\"(2 + 3) * 4\") == 20\n    assert safe_eval(\"7 // 2 + 7 % 2 - 2 ** 3\") == -4 and safe_eval(\"-x + +y\", {\"x\": 2, \"y\": 5}) == 3\n    assert safe_eval(\"1 / 4\") == 0.25 and safe_eval(\"0 < x < 10\", {\"x\": 3}) is True\n    assert safe_eval(\"x == 1 or y > 2\", {\"x\": 0, \"y\": 3}) is True and safe_eval(\"not (1 < 2)\") is False\n    assert safe_eval(\"big if n > 5 else small\", {\"n\": 9, \"big\": 1, \"small\": 0}) == 1\n    assert safe_eval(\"max(a, b) + abs(c)\", {\"a\": 1, \"b\": 4, \"c\": -2, \"max\": max, \"abs\": abs}) == 6\n    assert safe_eval(\"0 and boom\", {}) == 0 and safe_eval(\"1 or boom\", {}) == 1\n    assert safe_eval(\"True + 1\") == 2\n\ndef test_refused():\n    \"everything else\"\n    for bad in (\"__import__('os')\", \"x.real\", \"'text'\", \"[1, 2][0]\", \"(lambda: 1)()\", \"max(1, key=abs)\", \"2 ** 1000\"):\n        try:\n            safe_eval(bad, {\"x\": 1, \"max\": max, \"abs\": abs})\n            assert False, bad\n        except ValueError:\n            pass\n    try:\n        safe_eval(\"y + 1\", {})\n        assert False\n    except NameError:\n        pass\n    import re\n    assert not re.search(r\"(?<![_\\w])eval\\(\", source) and \"exec(\" not in source")

## Warm-up

Python turned the text into a tree of objects you can inspect (an `Assign` whose value is a `BinOp`), then compiled an expression into a **code object** that `eval` ran. Every program you've written has gone through these same steps.

## The pipeline

When you run `python shop.py`, CPython:

1. **reads** the file as text (UTF-8 by default);
2. **tokenises** it: splits the text into tokens such as names, numbers, operators and the special `INDENT` and `DEDENT` tokens that make indentation part of the grammar;
3. **parses** the tokens into an **abstract syntax tree** (AST): a tree of nodes like `Assign`, `Call` and `If`. Syntax errors are found here;
4. builds a **symbol table**, deciding for every name whether it's local, global, or a free variable of a closure (Week 60);
5. **compiles** the AST into **bytecode**: a compact list of instructions for a virtual machine, stored with its constants and names in a **code object**;
6. **evaluates** the code object in the **evaluation loop**, a large C function that reads one instruction at a time and carries it out (Wednesday).

Each stage has a tool in the standard library: `tokenize`, `ast`, `symtable`, `compile`, `dis` (tomorrow) and `exec`. The point of knowing them isn't to write a compiler, but to understand error messages, performance and tools: linters, formatters, test frameworks and coverage tools all work on these stages.

## Tokens

The **tokeniser** (or lexer) turns characters into tokens, each with a type, its text and its position. `tokenize.generate_tokens(readline)` exposes it: give it a function that returns one line at a time (`io.StringIO(text).readline`). Useful types:

- `NAME` (identifiers and keywords: `if` is a `NAME` token too), `NUMBER`, `STRING`, `OP` (operators and brackets);
- `NEWLINE` (the end of a logical line), `NL` (a blank line or a line break inside brackets), `COMMENT`;
- `INDENT` and `DEDENT`: Python's blocks, made by comparing each line's indentation with a stack of previous levels;
- `ENDMARKER` at the end.

Tokenising keeps comments and exact positions, which the AST throws away, so tools that must preserve your text (some formatters, syntax highlighters, `tokenize`-based refactoring tools) work at this level.

## The abstract syntax tree

`ast.parse(source)` returns a `Module` node whose `body` is a list of statements. Every node is an ordinary Python object with named fields (`node._fields` lists them) and, for most nodes, a position (`lineno`, `col_offset`). Some common nodes:

| code | node |
|---|---|
| `x = 1` | `Assign(targets=[Name('x')], value=Constant(1))` |
| `a + b` | `BinOp(left=Name('a'), op=Add(), right=Name('b'))` |
| `f(x)` | `Call(func=Name('f'), args=[Name('x')])` |
| `def f(): ...` | `FunctionDef(name='f', args=..., body=[...])` |
| `x` read / written | `Name(id='x', ctx=Load())` / `Name(id='x', ctx=Store())` |

Tools to work with trees:

- `ast.dump(node)` shows a node's structure; `ast.unparse(node)` turns a tree back into source code.
- `ast.walk(tree)` yields **every** node, in no particular order: fine for counting.
- **`ast.NodeVisitor`**: subclass it and define `visit_Name`, `visit_Call` and so on; call `visitor.visit(tree)`. Inside a `visit_X` method, call `self.generic_visit(node)` to carry on into the node's children, or they're skipped.
- **`ast.NodeTransformer`**: like a visitor, but each `visit_X` returns a replacement node (or the same node, or `None` to delete it). After changing a tree, call `ast.fix_missing_locations(tree)` so new nodes get positions before compiling.
- `ast.literal_eval(text)` safely evaluates text containing only literals (numbers, strings, lists, dicts, ...): unlike `eval`, it can't run code.

## compile and code objects

`compile(source, filename, mode)` turns source text (or an AST) into a **code object**. The `filename` is only used in error messages and tracebacks (`"<string>"` is the convention for code without a file). The mode says what kind of code it is:

- `"exec"`: any sequence of statements, run with `exec(code, namespace)`; the result is always `None`, and the statements' effects land in `namespace`;
- `"eval"`: a single **expression**, whose value `eval(code, namespace)` returns;
- `"single"`: one interactive statement, as the REPL uses (expression values are printed).

A code object is immutable and holds everything needed to run the code: `co_code` (the bytecode bytes), `co_consts` (constants), `co_names` (global and attribute names it uses), `co_varnames` (local variable names, arguments first), `co_argcount`, `co_freevars` (names from enclosing functions), `co_filename`, `co_firstlineno` and `co_name`. Every function holds one: `func.__code__`. Code objects for nested functions sit in the outer code object's `co_consts`; the `def` statement makes a **function object** from one each time it runs.

## Caching bytecode: .pyc files

Compiling takes time, so when you **import** a module, CPython saves the compiled code object to a `.pyc` file in a `__pycache__` folder next to the source, named after the interpreter version (`shop.cpython-313.pyc`). Next time, if the source hasn't changed (by default it compares the source's modification time and size, stored in the `.pyc` header), CPython loads the cached bytecode instead of recompiling.

Some consequences:

- The script you run directly (`python shop.py`) is compiled every time; only imported modules are cached.
- `.pyc` files are specific to a Python version, which is why the version is in the name, and why you never commit `__pycache__` to Git (Week 21's `.gitignore`).
- `python -m compileall folder` compiles a whole tree ahead of time; `importlib.util.cache_from_source(path)` tells you where the cached file for a source file would go.
- The `marshal` module is the format the code objects are saved in.

## Practice

## Quiz

## Daily challenge

## Before you go

Run `python -m ast your_file.py` (Python 3.9 and later) on a small file of your own to see its whole tree, and `python -m tokenize your_file.py` to see its tokens. Tomorrow: bytecode, the instructions inside a code object, and the `dis` module that shows them.